# DiffConvCubicBSpline

`DiffConvCubicBSpline` applies a cubic B-spline convolution that is differentiable with respect to the evaluation coordinates.
In effect, this produces a quasi-interpolant of the supplied data.

For instance, my research considers a form of convolutional physics-informed neural networks (PINNs) where the network is
conditioned by a coordinate-differentiable convolution of the data in the domain of dependence of the solution.  That
convolutional dependence amounts to the implementation of `DiffConvCubicBSpline` considered here.

There exists significant opportunity to extend the numerical and analytic notions of this form of differentiable convolution:
for example, there are many reasons one may wish to consider kernels other than the scaled cubic B-spline kernel used in this
layer.  Alternatively, one may wish to consider data on manifolds other than a Cartesian grid, such as irregularly sampled data
or even manifolds of dimension greater than zero.

In any case, in this notebook I hope to cover the basic usage of `DiffConvCubicBSpline`:
- Constructing and evaluating an interpolant.
- Differentiating discrete(!) data through means of the interpolant.

Before any of this, however, some setup:


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.func

from nnlayer import DiffConvCubicBSpline

# If you want to try higher precision, uncomment this:
# torch.set_default_dtype(torch.float64)

# Some plotting helpers:
FIGWIDTH = 8
FIGHEIGHT = 5
FIGDPI = 300
x_plot = torch.linspace(0, 2 * torch.pi, 1000)

## Constructing and Evaluating an Interpolant
Suppose we have some values ``f`` at points ``x``:


In [ ]:
N_SAMPLE = 20
N_EVAL = 1000

x = torch.linspace(0, 2 * torch.pi, N_SAMPLE)
dx = x[1] - x[0] 

f = lambda x: torch.sin(x) + torch.cos(4 * x)

plt.figure(figsize=(FIGWIDTH, FIGHEIGHT), dpi=FIGDPI)
plt.scatter(x, f(x), marker='.', label='Samples')
plt.plot(x_plot, f(x_plot))
plt.legend(loc='lower left')
plt.show()

Constructing the interpolant is easy and like any other torch layer:

In [ ]:
interpolant = DiffConvCubicBSpline(
    dx=dx.reshape((1,))  # (1,) corresponds to 1-d example here; dimension bookkeeping necessary for higher-d generalization.
)

To evaluate the interpolant, we supply three things:
- Coordinates ``x_data`` of the center cell of the convolution stencil.
- Data ``data`` of values in the convolution stencil.
- Evaluation coordinates ``x_eval``.  Note that ``x_eval`` must fall within the center cell of the convolution center.

Here's what that looks like in practice:

Because the support of the convolution can extend outside the boundary of the cell in which the evaluation
coordinate resides by at most ``dx`` in any dimension, the interpolation is defined on one cell less than
the Cartesian data grid in each direction in each dimension.

In 1-d, if data are defined on

$$
\Gamma = \{x_0, x_1, \dots, x_{N-2}, x_{N-1}\},
$$

then the interpolant is defined on

$$
U = [x_1, x_{N-2}].
$$

In 2-d, if data are defined on

$$
\Gamma = \{x_i\}_{i=0}^{N_x - 1} \times \{y_j\}_{j=0}^{N_y - 1},
$$

then the interpolant is defined on

$$
U = [x_1, x_{N_x-2}] \times [y_1, y_{N_y-2}].
$$

In general, if data are defined on the $N$-dimensional Cartesian grid,

$$
\Gamma = \prod_{k=1}^N \{ x^{(k)}_i \}_{i=0}^{N_k - 1}
$$

where $x^{(k)}_i$ is the $i^\text{th}$ coordinate in dimension $k$, then the interpolant is defined on

$$
U = \prod_{k=1}^N [x^{(k)}_1, x^{(k)}_{N_k - 2}].
$$

Per this logic, we compute a range of valid evaluation coordinates:


In [ ]:
x_eval = torch.linspace(x[1], x[-2], N_EVAL)
dx_eval = x_eval[1] - x_eval[0]

We compute indices in ``x`` where the coordinates ``x_eval`` reside and construct a view of ``x`` of these indices:

In [ ]:
indices = DiffConvCubicBSpline.get_nearest_index(x, x_eval)
x_data = x[indices]

The kernel in in ``DiffConvCubicBSpline`` is the product of one-dimensional cubic B-splines.  Each one-dimensional
spline is scaled to have support radius ``1`` in normalized grid-cell units.  Thus, an evaluation offset in
``[-0.5 dx, 0.5 dx]`` can overlap only the three cells centered at ``-1``, ``0``, and ``1`` along each coordinate
direction. For ``N`` coordinate dimensions, each evaluation thus uses a local ``3 ** N`` tensor-product
stencil.

We select data in this stencil:

In [ ]:
patch_indices = (
    indices[:, None]  # Index of stencil center, unsqueezed to broadcast correctly for dimension selection.
    + torch.arange(-1, 1 + 1)  # Indices about stencil center.
)
data = f(x)[patch_indices]

Finally, some dimension bookkeeping: this was necessary to allow (simple) generalization to higher dimensions:

In [ ]:
x_data = x_data.reshape((N_EVAL, 1))
x_eval = x_eval.reshape((N_EVAL, 1))

Now we can evaluate and plot the interpolant at all points ``x_eval``:

In [ ]:
f_interp = interpolant(  # Compute interpolation.
    x_data,
    data,
    x_eval
)

plt.figure(figsize=(FIGWIDTH, FIGHEIGHT), dpi=FIGDPI)
plt.scatter(x, f(x), marker='.', label='Samples')
plt.plot(x_plot, f(x_plot))
plt.plot(x_eval, f_interp, label='Interpolant')
plt.legend(loc='lower left')
plt.show()

## Differentiating the Interpolant
This is perhaps the most important capability of ``DiffConvCubicBSpline``: the result is differentiable with respect to the evaluation coordinates.


In [ ]:
partials = torch.func.vmap(torch.func.jacrev(interpolant, 2))(x_data, data, x_eval)

D = (torch.diag(-torch.ones(N_EVAL)) + torch.diag(torch.ones(N_EVAL - 1), 1)) / dx_eval
plt.figure(figsize=(FIGWIDTH, FIGHEIGHT), dpi=FIGDPI)
plt.scatter(x, f(x), marker='.', label='Samples')
plt.plot(x_plot, f(x_plot))
plt.plot(x_eval, f_interp, label='Interpolant')
plt.plot(x_plot.squeeze(), torch.func.vmap(torch.func.jacrev(f))(x_plot).squeeze(), label='Actual Derivative')
plt.plot(x_eval, partials.squeeze(), label='Derivative of Interpolant')
plt.legend(loc='lower left')
plt.grid()
plt.show()

## Higher Dimensions
We'll stick to 2-d for now, but the process is the same in any dimension.


In [ ]:
# Helpers for plotting and illustration:
x_plot, y_plot = torch.linspace(0, 2 * torch.pi, 1000), torch.linspace(-torch.pi, torch.pi, 1000)
xx_plot, yy_plot = torch.meshgrid(x_plot, y_plot)

Define and plot some underlying data, with sample points in red:

In [ ]:
N_COORDINATES = 2

# Sample coordinates:
x = torch.linspace(0, 2 * torch.pi, N_SAMPLE)
dx = x[1] - x[0] 

y = torch.linspace(-torch.pi, torch.pi, N_SAMPLE // 2)  # Changing it up a little.
dy = y[1] - y[0]

f = lambda x, y: torch.sin(x) + torch.cos(4 * x) * torch.cos(y)

xx, yy = torch.meshgrid(x, y)

plt.figure(figsize=(FIGWIDTH, FIGHEIGHT), dpi=FIGDPI)
plt.pcolormesh(xx_plot, yy_plot, f(xx_plot, yy_plot))
plt.scatter(xx, yy, color='tab:red', marker='.')
plt.colorbar()
plt.show()

In [ ]:
interpolant = DiffConvCubicBSpline(
    dx=torch.tensor((dx, dy))
)

x_eval = torch.linspace(x[1], x[-2], N_EVAL)
y_eval = torch.linspace(y[1], y[-2], N_EVAL)

xx_eval, yy_eval = torch.meshgrid(x_eval, y_eval)

x_indices = DiffConvCubicBSpline.get_nearest_index(x, x_eval)
y_indices = DiffConvCubicBSpline.get_nearest_index(y, y_eval)

Before proceeding, let me note that the indexing that follows is of more complexity than what would generally be required for straightforward evaluation of the interpolant at a sparse collection of points.  In particular, because I wish to evaluate the interpolant at all points in the meshgrid, there's some added complexity to the indexing that gets a little messy.

In [ ]:
xx_indices, yy_indices = torch.meshgrid(x_indices, y_indices)

coord_data = torch.stack(
    (x[xx_indices], y[yy_indices]),
    dim=-1
)
coord_eval = torch.stack(
    (xx_eval, yy_eval),
    dim=-1
)

index_num_shape = lambda n: (*((1,) * n), 3, *((1,) * (N_COORDINATES - n - 1)))
index_num_repeats = lambda n: (*((3,) * n), 1, *((3,) * (N_COORDINATES - n - 1)))

_index_num = 0
patch_xx_indices = torch.reshape(xx_indices, (*xx_indices.shape, *((1,) * N_COORDINATES)))
patch_xx_indices = patch_xx_indices.repeat(*((1,) * xx_indices.ndim), *((3,) * N_COORDINATES))
patch_xx_indices = patch_xx_indices + torch.reshape(torch.arange(-1, 1 + 1), index_num_shape(_index_num)).repeat(index_num_repeats(_index_num))

_index_num = 1
patch_yy_indices = torch.reshape(yy_indices, (*yy_indices.shape, *((1,) * N_COORDINATES)))
patch_yy_indices = patch_yy_indices.repeat(*((1,) * yy_indices.ndim), *((3,) * N_COORDINATES))
patch_yy_indices = patch_yy_indices + torch.reshape(torch.arange(-1, 1 + 1), index_num_shape(_index_num)).repeat(index_num_repeats(_index_num))

data = f(xx, yy)[patch_xx_indices, patch_yy_indices]


We can compute an evaluation of the interpolant, and we ought to plot it to compare against the real function:

In [ ]:
f_interp = interpolant(
    coord_data,
    data,
    coord_eval
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(FIGWIDTH, 2 * FIGHEIGHT), dpi=FIGDPI, sharex=True, sharey=True)
ax1.set_title('True Function')
m = ax1.pcolormesh(xx_plot, yy_plot, f(xx_plot, yy_plot), vmax=2., vmin=-2)
plt.colorbar(m, ax=ax1)
ax2.set_title('Interpolant')
m = ax2.pcolormesh(xx_eval, yy_eval, f_interp, vmax=2., vmin=-2)
plt.colorbar(m, ax=ax2)
plt.show()


Of course, we can compute the gradient as well:

In [ ]:
# Nested vmap to peel off both batch dimensions (in this case corresponding to spatial dimensions):
interpolant_partials = torch.func.vmap(torch.func.vmap(torch.func.jacrev(interpolant, 2)))(coord_data, data, coord_eval)

# Different structure of f requires computing partials differently:
f_partials = torch.stack(
    (
        torch.func.vmap(torch.func.vmap(torch.func.jacrev(f, 0)))(xx_eval, yy_eval),
        torch.func.vmap(torch.func.vmap(torch.func.jacrev(f, 1)))(xx_eval, yy_eval)
    ),
    dim=-1
)

N_QUIVERS_X = 16
N_QUIVERS_Y = 10

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(FIGWIDTH, 2 * FIGHEIGHT), dpi=FIGDPI, sharex=True, sharey=True)
ax1.set_title('True Function')
m = ax1.pcolormesh(xx_plot, yy_plot, f(xx_plot, yy_plot), vmax=2., vmin=-2)
plt.colorbar(m, ax=ax1)
ax1.quiver(
    xx_eval[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y],
    yy_eval[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y],
    f_partials[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y, 0],
    f_partials[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y, 1]
)
ax2.set_title('Interpolant')
m = ax2.pcolormesh(xx_eval, yy_eval, f_interp, vmax=2., vmin=-2)
plt.colorbar(m, ax=ax2)
ax2.quiver(
    xx_eval[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y],
    yy_eval[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y],
    interpolant_partials[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y, 0],
    interpolant_partials[::N_EVAL // N_QUIVERS_X, ::N_EVAL // N_QUIVERS_Y, 1]
)
plt.show()

Finally, we ought to compare the error of the interpolant and its gradient:

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(FIGWIDTH, 2 * FIGHEIGHT), dpi=FIGDPI, sharex=True, sharey=True)
ax1.set_title('Error')
m = ax1.pcolormesh(xx_eval, yy_eval, f(xx_eval, yy_eval) - f_interp)
plt.colorbar(m, ax=ax1, label='$f_\\text{true} - f_\\text{interpolant}$')
ax1.set_xlim(min(x), max(x))
ax1.set_ylim(min(y), max(y))
ax2.set_title('Gradient Error')
m = ax2.pcolormesh(xx_eval, yy_eval, torch.sqrt(torch.sum((f_partials - interpolant_partials) ** 2, dim=-1)))
plt.colorbar(m, ax=ax2, label='$\\| \\nabla f_\\text{true} - \\nabla f_\\text{interpolant}\\|$')
ax2.set_xlim(min(x), max(x))
ax2.set_ylim(min(y), max(y))
plt.show()

## Gotchas
Here are some of the especially tricky bits about this convolution layer:

### Slow Gradients if Not Careful
If not carefully using torch's grad facilities, it's easy to implement something that's much slower than it ought to be.

Consider again our 1-d test case:

In [ ]:
x = torch.linspace(0, 2 * torch.pi, N_SAMPLE)
dx = x[1] - x[0] 
f = lambda x: torch.sin(x) + torch.cos(4 * x)

x_eval = torch.linspace(x[1], x[-2], N_EVAL)
dx_eval = x_eval[1] - x_eval[0]

indices = DiffConvCubicBSpline.get_nearest_index(x, x_eval)

x_data = x[indices].reshape((N_EVAL, 1))
data = f(x)[indices[:, None] + torch.arange(-1, 1 + 1)]
x_eval = x_eval.reshape((N_EVAL, 1))

interpolant = DiffConvCubicBSpline(dx=dx.reshape((1,)))

How fast is computing a gradient with respect to the coordinate ``x``?

Let's try an unwrapped ``jacrev``:

In [ ]:
%timeit torch.func.jacrev(interpolant, 2)(x_data, data, x_eval)

Let's try the same, but with ``vmap``:

In [ ]:
%timeit torch.func.vmap(torch.func.jacrev(interpolant, 2))(x_data, data, x_eval)

That's a massive improvement for a ``vmap`` call.  I'll also note that this is effectively the textbook use-case of ``vmap`` (that is, at least, my understanding).

What's going on here is that the un``vmap``ed ``jacrev`` call is computing the Jacobian of the interpolant with respect to all inputs, even though one index of the result varies with only one index of the input: the naked ``jacrev`` is computing a Jacobian of size ``N_INTERP``$\times$``N_INTERP``, while the ``vmap`` call is computing a $1 \times 1$ Jacobian ``N_INTERP`` times.

**Bottom line:** compute only what you have to.
